In [10]:
import pulp
import pandas as pd

In [11]:
df = pd.read_csv('../data/processed/rq3_inputs.csv')

In [12]:
BATTERY_CAPACITY = 500
MIN_SOC = 0.1 * BATTERY_CAPACITY
MAX_SOC = 0.9 * BATTERY_CAPACITY
MAX_CHARGE_RATE = 100
MAX_DISCHARGE_RATE = 100
CHARGE_EFFICIENCY = 0.95
DISCHARGE_EFFICIENCY = 0.95

In [13]:
def solve_mpc_window(prices, productions, initial_soc, capacity, min_soc, max_soc,
                      max_charge, max_discharge, charge_eff, discharge_eff):
    H = len(prices)
    prob = pulp.LpProblem("battery_mpc", pulp.LpMaximize)

    sell = [pulp.LpVariable(f"sell_{t}", lowBound=0) for t in range(H)]
    charge = [pulp.LpVariable(f"charge_{t}", lowBound=0, upBound=max_charge) for t in range(H)]
    discharge = [pulp.LpVariable(f"discharge_{t}", lowBound=0, upBound=max_discharge) for t in range(H)]
    soc = [pulp.LpVariable(f"soc_{t}", lowBound=min_soc, upBound=max_soc) for t in range(H)]

    prob += pulp.lpSum([prices[t] * (sell[t] + discharge[t] * discharge_eff) / 1000 for t in range(H)])

    for t in range(H):
        prob += sell[t] + charge[t] == productions[t]

        if t == 0:
            prob += soc[t] == initial_soc + charge[t] * charge_eff - discharge[t]
        else:
            prob += soc[t] == soc[t-1] + charge[t] * charge_eff - discharge[t]

    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    results = [(sell[t].varValue, charge[t].varValue, discharge[t].varValue) for t in range(H)]
    return results

In [14]:
HORIZON = 24
soc_actual = MIN_SOC
revenue_mpc = 0
soc_history_mpc = []
actions_log = []

predicted_prices = df['predicted_price'].values
actual_prices = df['actual_price'].values
predicted_P = df['predicted_P'].values
actual_P = df['actual_P'].values

n = len(df)

for t in range(n):
    window_end = min(t + HORIZON, n)
    window_len = window_end - t

    window_prices = predicted_prices[t:window_end]
    window_prod = predicted_P[t:window_end]

    decisions = solve_mpc_window(
        window_prices, window_prod, soc_actual,
        BATTERY_CAPACITY, MIN_SOC, MAX_SOC,
        MAX_CHARGE_RATE, MAX_DISCHARGE_RATE,
        CHARGE_EFFICIENCY, DISCHARGE_EFFICIENCY
    )

    sell_amt, charge_amt, discharge_amt = decisions[0]

    real_price = actual_prices[t]
    real_production = actual_P[t]

    actual_sell = min(sell_amt, real_production)
    actual_charge = min(charge_amt, real_production - actual_sell, MAX_SOC - soc_actual)

    revenue_mpc += real_price * (actual_sell + discharge_amt * DISCHARGE_EFFICIENCY) / 1000
    soc_actual = soc_actual + actual_charge * CHARGE_EFFICIENCY - discharge_amt

    soc_history_mpc.append(soc_actual)
    actions_log.append({'sell': actual_sell, 'charge': actual_charge, 'discharge': discharge_amt})

print(f"Вкупен приход (MPC): {revenue_mpc:.2f} EUR")
print(f"Финална состојба: {soc_actual:.2f} kWh")

Вкупен приход (MPC): 1111.54 EUR
Финална состојба: 50.00 kWh


In [17]:
soc_actual_pf = MIN_SOC
revenue_perfect = 0

for t in range(n):
    window_end = min(t + HORIZON, n)
    window_prices = actual_prices[t:window_end]
    window_prod = actual_P[t:window_end]

    decisions = solve_mpc_window(
        window_prices, window_prod, soc_actual_pf,
        BATTERY_CAPACITY, MIN_SOC, MAX_SOC,
        MAX_CHARGE_RATE, MAX_DISCHARGE_RATE,
        CHARGE_EFFICIENCY, DISCHARGE_EFFICIENCY
    )

    sell_amt, charge_amt, discharge_amt = decisions[0]
    real_price = actual_prices[t]

    revenue_perfect += real_price * (sell_amt + discharge_amt * DISCHARGE_EFFICIENCY) / 1000
    soc_actual_pf = soc_actual_pf + charge_amt * CHARGE_EFFICIENCY - discharge_amt

print(f"Вкупен приход (Perfect Foresight): {revenue_perfect:.2f} EUR")

Вкупен приход (Perfect Foresight): 1246.60 EUR
